# 基于transformer实现的中译英模型

## mount google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## import

In [ ]:
import codecs
import os
import numpy as np
import requests
import regex
import jieba
from typing import Optional
import time
from collections import Counter
import random
import math

import torch
from torch import nn
import torch.nn.functional as F

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")


## check GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cuda")
!nvidia-smi

Thu Jun 11 14:46:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## data load

In [ ]:
url = "https://object.pouta.csc.fi/OPUS-OpenSubtitles/v2018/moses/en-zh_cn.txt.zip"
zip_name = "en-zh_cn.txt.zip"

if not os.path.exists(zip_name):
  print("Downloading from OPUS...")
  os.system(f"wget {url}")
  os.system(f"unzip {zip_name}")

def limit_corpus(input_file, output_file, max_lines=500000):
  print(f"processing {input_file}, saving the front {max_lines} lines")
  with open(input_file, 'r', encoding='utf-8') as f_in, \
     open(output_file, 'w', encoding='utf-8') as f_out:
     for i, line in enumerate(f_in):
      if i >= max_lines:
        break
      f_out.write(line)
  print(f"saving successfully")

limit_corpus('OpenSubtitles.en-zh_cn.en', './en.txt', max_lines=500000)
limit_corpus('OpenSubtitles.en-zh_cn.zh_cn', './cn.txt', max_lines=500000)

processing OpenSubtitles.en-zh_cn.en, saving the front 500000 lines
saving successfully
processing OpenSubtitles.en-zh_cn.zh_cn, saving the front 500000 lines
saving successfully


## data process

In [ ]:
max_len = 50 # maximum number in a sentence


def load_vocab(language, max_vocab_size=30000):
    assert language in ['cn', 'en']
    counter = Counter()
    with open(f"./{language}.txt", 'r', encoding='utf-8') as f:
      for line in f:
        if language == 'cn':
          words = jieba.lcut(line.strip())
        else:
          words = line.strip().lower().split()
        counter.update(words)
    most_common = counter.most_common(max_vocab_size)
    special = ['<PAD>', '<UNK>', '<S>', '</S>']
    vocab = special + [item[0] for item in most_common]
    word2idx = {word : i for i, word in enumerate(vocab)}
    idx2word = [word for word in vocab]
    return word2idx, idx2word


def create_data(source_sents, target_sents):
    cn2idx, idx2cn = load_vocab('cn')
    en2idx, idx2en = load_vocab('en')
    # idx
    x_list, y_list, sources, targets = [], [], [], []
    for source_sent, target_sent in zip(source_sents, target_sents):
        x = [
            cn2idx.get(w, 1)
            for w in (['<S>'] + jieba.lcut(source_sent) + ['</S>'])
        ]
        y = [
            en2idx.get(w, 1)
            for w in (['<S>'] + target_sent.lower().split() + ['</S>'])
        ]
        if max(len(x), len(y)) < max_len and len(x) > 2 and len(y) > 2:
            x_list.append(np.array(x))
            y_list.append(np.array(y))
            sources.append(source_sent)
            targets.append(target_sent)
    # pad
    X = np.zeros([len(x_list), max_len], np.int32)
    Y = np.zeros([len(y_list), max_len], np.int32)
    for i, (x, y) in enumerate(zip(x_list, y_list)):
        X[i] = np.pad(x, (0, max_len - len(x)), 'constant', constant_values=(0,0))
        Y[i] = np.pad(y, (0, max_len - len(y)), 'constant', constant_values=(0,0))
    return X, Y, sources, targets


def load_data(type):
    assert type in ['train', 'test']
    if type == 'train':
        source, target = 'cn.txt', 'en.txt'
    else:
        # source, target = 'cn.test.txt', 'en.test.txt'
        assert(False), ("Waiting for adding")
    cn_sents = [
        regex.sub("[^\s\p{L}']", '', line)
        for line in codecs.open(source, 'r', 'utf-8').read().splitlines()
        if line and line[0] != '<'
    ]
    en_sents = [
        regex.sub("[^\s\p{L}']", '', line)
        for line in codecs.open(target, 'r', 'utf-8').read().splitlines()
        if line and line[0] != '<'
    ]
    X, Y, source, target = create_data(cn_sents, en_sents)
    return X, Y, source, target


def get_batch_indices(total_length, batch_size):
    assert (batch_size <= total_length), ('Batch size is larger than data length.')
    indexs = list(range(total_length))
    random.shuffle(indexs)
    for i in range(0, total_length, batch_size):
      batch = indexs[i : i + batch_size]
      yield batch

def idx2sentence(arr, vocab, insert_space=False):
    res = ''
    first_word = True
    for id in arr:
        word = vocab[id.item()]
        if insert_space and not first_word: # insert space between words
            res += ' '
        first_word = False
        res += word
    return res


<>:58: SyntaxWarning: invalid escape sequence '\s'
<>:63: SyntaxWarning: invalid escape sequence '\s'
<>:58: SyntaxWarning: invalid escape sequence '\s'
<>:63: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_761/661792808.py:58: SyntaxWarning: invalid escape sequence '\s'
  regex.sub("[^\s\p{L}']", '', line)
/tmp/ipykernel_761/661792808.py:63: SyntaxWarning: invalid escape sequence '\s'
  regex.sub("[^\s\p{L}']", '', line)


## Constant

In [ ]:
batch_size = 64
lr = 1e-5
d_model = 512 # embedding length
d_ff = 2048
layers = 6
heads = 8
dropout_rate = 0.1
epochs = 160
PAD_ID = 0

## model


### positional encoding

In [ ]:
class PositionalEncoding(nn.Module): ###
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        assert d_model % 2 == 0 # assume even for convenience
        i_seq = torch.linspace(0, max_len - 1, max_len) # generate list
        j_seq = torch.linspace(0, d_model - 2, d_model // 2)
        pos, two_i = torch.meshgrid(i_seq, j_seq)
        pe_2i = torch.sin(pos / 10000**(two_i / d_model))
        pe_2i_1 = torch.cos(pos / 10000**(two_i / d_model))
        pe = torch.stack((pe_2i, pe_2i_1), dim=2).reshape(1, max_len, d_model)
        self.register_buffer('pe', pe, persistent=False)

    def forward(self, x):
        n, seq_len, d_model = x.shape
        pe: torch.Tensor = self.pe
        assert seq_len <= pe.shape[1]
        assert d_model == pe.shape[2]
        rescaled_x = x * d_model**0.5
        return rescaled_x + pe[:, 0:seq_len, :]

### attention

In [ ]:
def attention(q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        mask: Optional[torch.Tensor] = None):
    assert q.shape[-1] == k.shape[-1]
    d_k = k.shape[-1]
    # tmp shape: [n, heads, q_len, k_len]
    tmp = torch.matmul(q, k.transpose(-2, -1)) / d_k**0.5
    if mask is not None:
        min_value = -1e4
        tmp.masked_fill_(mask, min_value)
    tmp = F.softmax(tmp, dim=-1) + 1e-9 # 防止softmax输出全0导致nan
    # tmp.shape: [n, heads, q_len, d_v]
    tmp = torch.matmul(tmp, v)
    return tmp

### multi-head attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % heads == 0
        self.d_k = d_model // heads
        self.heads = heads
        self.d_model = d_model
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self,
          q: torch.Tensor, k: torch.Tensor,
          v: torch.Tensor, mask: Optional[torch.Tensor] = None):
        # batch size
        assert q.shape[0] == k.shape[0]
        assert q.shape[0] == v.shape[0]
        # sequence length of k and v should be equaled
        assert k.shape[1] == v.shape[1]
        n, q_len = q.shape[0:2]
        n, k_len = k.shape[0:2]
        q_ = self.q(q).reshape(n, q_len, self.heads, self.d_k).transpose(1, 2)
        k_ = self.k(k).reshape(n, k_len, self.heads, self.d_k).transpose(1, 2)
        v_ = self.v(v).reshape(n, k_len, self.heads, self.d_k).transpose(1, 2)
        attention_res = attention(q_, k_, v_, mask)
        concat_res = attention_res.transpose(1, 2).reshape(
            n, q_len, self.d_model)
        concat_res = self.dropout(concat_res)
        output = self.out(concat_res)
        return output

### feed forward

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.layer1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.layer2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.layer1(x)
        x = self.dropout(F.relu(x))
        x = self.layer2(x)
        return x

### Encoder

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self,
                 d_model: int,
                 heads: int,
                 d_ff: int,
                 dropout: float = 0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask: Optional[torch.Tensor] = None):
        tmp = self.self_attention(x, x, x, src_mask)
        tmp = self.dropout1(tmp)
        x = self.norm1(tmp + x)
        tmp = self.ffn(x)
        tmp = self.dropout2(tmp)
        x = self.norm2(tmp + x)
        return x


class Encoder(nn.Module):
    def __init__(self,
                 vocab_size: int,
                 pad_idx: int,
                 d_model: int,
                 d_ff: int,
                 heads: int,
                 dropout: float = 0.1,
                 n_layers: int = 6,
                 max_seq_len: int = 120):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, pad_idx)
        self.pe = PositionalEncoding(d_model, max_seq_len)
        self.layers = []
        for i in range(n_layers):
            self.layers.append(EncoderLayer(d_model, heads, d_ff, dropout))
        self.layers = nn.ModuleList(self.layers)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask: Optional[torch.Tensor] = None):
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pe(x)
        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return x


### decoder

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self,
                 d_model: int,
                 heads: int,
                 d_ff: int,
                 dropout: float = 0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, heads, dropout)
        self.attention = MultiHeadAttention(d_model, heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self,
                x,
                encoder_kv: torch.Tensor,
                dst_mask: Optional[torch.Tensor] = None,
                src_dst_mask: Optional[torch.Tensor] = None):
        tmp = self.self_attention(x, x, x, dst_mask)
        tmp = self.dropout1(tmp)
        x = self.norm1(tmp + x)
        tmp = self.attention(x, encoder_kv, encoder_kv, src_dst_mask)
        tmp = self.dropout2(tmp)
        x = self.norm2(tmp + x)
        tmp = self.ffn(x)
        tmp = self.dropout3(tmp)
        x = self.norm3(tmp + x)
        return x


class Decoder(nn.Module):
    def __init__(self,
                 vocab_size: int,
                 pad_idx: int,
                 d_model: int,
                 d_ff: int,
                 heads: int,
                 dropout: float = 0.1,
                 n_layers: int = 6,
                 max_seq_len: int = 120):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, pad_idx)
        self.pe = PositionalEncoding(d_model, max_seq_len)
        self.layers = []
        for i in range(n_layers):
            self.layers.append(DecoderLayer(d_model, heads, d_ff, dropout))
        self.layers = nn.Sequential(*self.layers)
        self.dropout = nn.Dropout(dropout)

    def forward(self,
                x,
                encoder_kv: torch.Tensor,
                dst_mask: Optional[torch.Tensor] = None,
                src_dst_mask: Optional[torch.Tensor] = None):
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pe(x)
        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x, encoder_kv, dst_mask, src_dst_mask)
        return x

### transformer

In [ ]:
class Transformer(nn.Module):
    def __init__(self,
                 src_vocab_size: int,
                 dst_vocab_size: int,
                 pad_idx: int,
                 d_model: int,
                 d_ff: int,
                 n_layers: int,
                 heads: int,
                 dropout: float = 0.1,
                 max_seq_len: int = 120):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, pad_idx, d_model, d_ff,
                               heads, dropout, n_layers, max_seq_len)
        self.decoder = Decoder(dst_vocab_size, pad_idx, d_model, d_ff,
                               heads, dropout, n_layers, max_seq_len)
        self.pad_idx = pad_idx
        self.output_layer = nn.Linear(d_model, dst_vocab_size)
        # parameters initialize
        self._init_weights()

    def _init_weights(self):
          for p in self.parameters():
              if p.dim() > 1:
                  nn.init.xavier_uniform_(p)

    def generate_mask(self,
              q_pad: torch.Tensor, # [n, q_len]
              k_pad: torch.Tensor, # [n, k_len]
              with_left_mask: bool = False):
        assert q_pad.device == k_pad.device
        n, q_len = q_pad.shape
        n, k_len = k_pad.shape

        mask = k_pad[:, None, None, :]

        if with_left_mask:
            # [1, 1, q_len, k_len]
            left_mask = torch.triu(torch.ones((1, 1, q_len, k_len), device=q_pad.device), diagonal=1).bool()
            mask = mask | left_mask
        return mask

    def forward(self, x, y):
        src_pad_mask = x == self.pad_idx
        dst_pad_mask = y == self.pad_idx
        src_mask = self.generate_mask(src_pad_mask, src_pad_mask, False)
        dst_mask = self.generate_mask(dst_pad_mask, dst_pad_mask, True)
        src_dst_mask = self.generate_mask(dst_pad_mask, src_pad_mask, False)
        encoder_kv = self.encoder(x, src_mask)
        res = self.decoder(y, encoder_kv, dst_mask, src_dst_mask)
        res = self.output_layer(res)
        return res

## train

In [ ]:
import matplotlib.pyplot as plt
import math

def train():
    print('\n====== Training ======')
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = False
    # scaler = torch.cuda.amp.GradScaler()

    cn2idx, idx2cn = load_vocab('cn')
    en2idx, idx2en = load_vocab('en')
    X, Y, source, target = load_data('train')

    for i in range(10):
      print(source[i])
      print(target[i])

    print(len(cn2idx))
    print(len(en2idx))

    model = Transformer(len(cn2idx), len(en2idx), PAD_ID, d_model,
              d_ff, layers, heads, dropout_rate, max_len)
    model.to(device)
    print(model)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.5,
        betas=(0.9, 0.98),
        eps=1e-9
    )

    criterion = nn.CrossEntropyLoss(
        ignore_index=PAD_ID,
        label_smoothing=0.0
    ) # ignore the <PAD>
    start = time.time()
    train_losses = []
    total_steps = 0
    start_epoch = 0
    best_loss = float('inf')

    def noam_lambda(step):
      warmup = 8000
      step = step + 1
      return (d_model ** -0.5) * min(
          step ** -0.5,
          step * warmup ** -1.5
      )
    # scheduler = torch.optim.lr_scheduler.LambdaLR(
    #     optimizer,
    #     lr_lambda=noam_lambda
    # )

    num_batches = (len(X) + batch_size - 1) // batch_size

    checkpoint_path = './drive/MyDrive/transformer_checkpoint/checkpoint.pth'
    bestmodel_path = './drive/MyDrive/transformer_checkpoint/best_model.pth'
    if os.path.exists(checkpoint_path):
      print('find the checkpoint for continue training')
      checkpoint = torch.load(checkpoint_path)
      model.load_state_dict(checkpoint['model_state_dict'])
      optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
      # scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
      # scaler.load_state_dict(checkpoint['scaler_state_dict'])
      start_epoch = checkpoint['epoch']
      print(f'start from epoch {start_epoch + 1}')

    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss = 0
        X_tensor = torch.from_numpy(X).long()
        Y_tensor = torch.from_numpy(Y).long()
        print(f"=== {epoch + 1} ===")
        for index in get_batch_indices(len(X), batch_size):
            x_batch = X_tensor[index].to(device, non_blocking=True)
            y_batch = Y_tensor[index].to(device, non_blocking=True)
            y_input = y_batch[:, :-1] # offer the first n-1 words(<S> A B C)
            y_label = y_batch[:, 1:] # use 2-n words as label(A B C </S>)

            # with torch.amp.autocast('cuda'):
            y_hat = model(x_batch, y_input)
            n, seq_len = y_label.shape
            y_hat = y_hat.reshape(n * seq_len, -1)
            y_label = y_label.reshape(n * seq_len)

            loss = criterion(y_hat, y_label)
            optimizer.zero_grad(set_to_none=True)

            # scaler.scale(loss).backward()
            # scaler.unscale_(optimizer)
            # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            # scaler.step(optimizer)
            # scaler.update()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            # scheduler.step()   # ← Noam 每 step

            train_loss += loss.item()
            total_steps += 1
            if total_steps % 500 == 0:
                end = time.time()
                print(f'Time:{end - start:.4f}s Loss:{loss.item():.4f}')

        train_losses.append(train_loss)
        avg_loss = train_loss / num_batches
        # if (epoch + 1) % 2 == 0:
        checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                # 'scheduler_state_dict': scheduler.state_dict(),
                # 'scaler_state_dict': scaler.state_dict()
                }
        torch.save(checkpoint, checkpoint_path)
        print(f'save the model at epoch {epoch + 1}')

        if avg_loss < best_loss:
          best_loss = avg_loss
          torch.save(checkpoint, bestmodel_path)
          print(f"--- Best model saved (Loss: {best_loss:.4f}) ---")


    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(epochs, train_losses, label='Train Loss')
    plt.title('Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.show()

    print('\n====== Saving model ======')
    model_path = './drive/MyDrive/transformer_checkpoint/model.pth'
    torch.save(model.state_dict(), model_path)

In [ ]:
train()


====== Training ======
好想要吃泡菜
Ah this is greasy I want to eat kimchee
崔允的造型师在吗
Is Chae Yoon's coordinator in here
请问一下 你是不是崔允的造型师
Excuse me aren't you Chae Yoon's coordinator
我吗
Yes Me
崔允已经唱完了
Chae Yoon is done singing
这位小姐是谁 我想大家都知道吧
This lady right next to me everyone knows who she is right
她是我永远的偶像 也是我无法击倒的对手 她是令我怦然心动的恋人
She is my forever idolshe is my rival whom I can never win against she is the one who always makes me happy and she is my mother
也许在各位的记忆里 我母亲 她是曾经红极一时的女演员
Everyone should remember my mom as one of the greatest movie stars but instead of becoming an actress she wanted to become a singer just like me
而是像我一样当歌手 当我感到累还有疲倦的时候 每一次她都让我
When I was stressed and exhausted she'd always sing me a song while having me on her lap
然后为我唱一首歌 今天我想要 特别和各位
Today I would like to share that moment with everyone
30004
30004


## translate

In [ ]:
def main():

    cn2idx, idx2cn = load_vocab('cn')
    en2idx, idx2en = load_vocab('en')

    model = Transformer(
        len(cn2idx),
        len(en2idx),
        PAD_ID,
        d_model,
        d_ff,
        layers,
        heads,
        dropout_rate,
        max_len
    ).to(device)

    checkpoint = torch.load(
        './drive/MyDrive/transformer_checkpoint/best_model.pth',
        map_location=device
    )

    model.load_state_dict(
        checkpoint['model_state_dict']
    )

    model.eval()

    sentence = "这部电影很不错"
    UNK_ID = cn2idx['<UNK>']

    tokens = [
        cn2idx.get(w, UNK_ID)
        for w in jieba.cut(sentence)
    ]

    x_batch = torch.LongTensor(tokens)\
        .unsqueeze(0)\
        .to(device)

    y_input = torch.full(
        (1, max_len),
        PAD_ID,
        dtype=torch.long
    ).to(device)

    y_input[0, 0] = en2idx['<S>']

    with torch.no_grad():

        for i in range(1, max_len):

            y_hat = model(x_batch, y_input)

            next_token = torch.argmax(
                y_hat[0, i - 1]
            )

            y_input[0, i] = next_token

            if next_token == en2idx['</S>']:
                break

    output_sentence = idx2sentence(
        y_input[0],
        idx2en,
        True
    )

    print(output_sentence)
    # tokens = list(jieba.cut(sentence))
    # for t in tokens:
    #     print(
    #         t,
    #         t in cn2idx
    #     )

if __name__ == '__main__':
    main()